# LeetCode 1744 — Can You Eat Your Favorite Candy on Your Favorite Day? bằng Python

Link đề: https://leetcode.com/problems/can-you-eat-your-favorite-candy-on-your-favorite-day/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def canEat(self, candiesCount: List[int], queries: List[List[int]]) -> List[bool]:
        ...
```

Lộ trình: **vét cạn mọi lịch ăn** (tập trạng thái theo ngày, chỉ cho input nhỏ) → **mỗi truy vấn cộng dồn lại từ đầu** `O(n)`/truy vấn → **tổng tiền tố + giao hai khoảng** `O(1)`/truy vấn → biến thể **tìm kiếm nhị phân** trên tổng tiền tố. Mấu chốt: tập "số thứ tự viên kẹo có thể ăn vào ngày `d`" là **một khoảng liên tục**.

## 1. Mục tiêu học

1. Chuyển một trò chơi có luật (ăn theo thứ tự loại, ít nhất 1 / nhiều nhất `cap` mỗi ngày) thành bài toán **khoảng số**: đánh số mọi viên kẹo `0, 1, 2, …` theo thứ tự phải ăn.
2. Dùng **tổng tiền tố** `s[t]` để biết loại `t` chiếm đoạn viên kẹo `[s[t], s[t+1]-1]`.
3. Lập luận **trường hợp chậm nhất / nhanh nhất** (1 viên/ngày vs `cap` viên/ngày) và chứng minh mọi giá trị ở giữa đều đạt được.
4. Kiểm tra **hai khoảng có giao nhau** bằng điều kiện `lo1 ≤ hi2 and lo2 ≤ hi1`, cẩn thận lệch-1 với ngày đánh số từ 0.
5. Viết vét cạn bằng **tập trạng thái theo ngày** để đối chiếu công thức trên input ngẫu nhiên.

## 2. Đề bài & ví dụ

Cho mảng số nguyên dương `candiesCount` (đánh số từ 0), `candiesCount[i]` là số kẹo loại `i`. Cho mảng `queries`, mỗi `queries[i] = [favoriteType, favoriteDay, dailyCap]`.

Luật chơi:
- Bắt đầu ăn từ ngày **0**.
- **Không** được ăn kẹo loại `i` khi chưa ăn **hết** kẹo loại `i - 1`.
- Mỗi ngày phải ăn **ít nhất một** viên cho tới khi ăn hết kẹo.

Với mỗi truy vấn, `answer[i] = true` nếu **có cách** ăn được (ít nhất một viên) kẹo loại `favoriteType` vào ngày `favoriteDay` mà **không ngày nào** ăn quá `dailyCap` viên. Một ngày có thể ăn nhiều loại khác nhau (miễn tuân luật 2).

**Ví dụ 1:** `candiesCount = [7,4,5,3,8]`, `queries = [[0,2,2],[4,2,4],[2,13,1000000000]]` → `[true,false,true]`.
- Ăn 2 viên ngày 0, 2 viên ngày 1 → ngày 2 vẫn còn kẹo loại 0.
- Tối đa 4 viên/ngày: ngày 0 ăn 4 (loại 0), ngày 1 ăn 4 (loại 0, 1), ngày 2 chỉ tới loại 1–2 → không chạm loại 4.
- Ăn 1 viên/ngày → ngày 13 ăn viên thứ 14, thuộc loại 2.

**Ví dụ 2:** `candiesCount = [5,2,6,4,1]`, `queries = [[3,1,2],[4,10,3],[3,10,100],[4,100,30],[1,3,1]]` → `[false,true,true,false,false]`.

**Ràng buộc:** `1 ≤ candiesCount.length ≤ 10⁵`, `1 ≤ candiesCount[i] ≤ 10⁵`, `1 ≤ queries.length ≤ 10⁵`, `0 ≤ favoriteType < candiesCount.length`, `0 ≤ favoriteDay ≤ 10⁹`, `1 ≤ dailyCap ≤ 10⁹`.

In [1]:
import random
import time
from bisect import bisect_right
from itertools import accumulate
from typing import List

examples = [
    ([7, 4, 5, 3, 8], [[0, 2, 2], [4, 2, 4], [2, 13, 1000000000]], [True, False, True]),
    ([5, 2, 6, 4, 1], [[3, 1, 2], [4, 10, 3], [3, 10, 100], [4, 100, 30], [1, 3, 1]],
     [False, True, True, False, False]),
]
for c, q, want in examples:
    print("candiesCount =", c)
    for qq, w in zip(q, want):
        print("   query", qq, "-> kỳ vọng", w)

candiesCount = [7, 4, 5, 3, 8]
   query [0, 2, 2] -> kỳ vọng True
   query [4, 2, 4] -> kỳ vọng False
   query [2, 13, 1000000000] -> kỳ vọng True
candiesCount = [5, 2, 6, 4, 1]
   query [3, 1, 2] -> kỳ vọng False
   query [4, 10, 3] -> kỳ vọng True
   query [3, 10, 100] -> kỳ vọng True
   query [4, 100, 30] -> kỳ vọng False
   query [1, 3, 1] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1 Tổng tiền tố với `itertools.accumulate(..., initial=0)`

`s[i]` = tổng `candiesCount[0..i-1]` (tổng của `i` phần tử đầu). Có `initial=0` thì `s[0] = 0` và `len(s) = n + 1` — rất tiện vì đoạn `[l, r)` có tổng `s[r] - s[l]`.

In [2]:
cnt = [7, 4, 5, 3, 8]
s = list(accumulate(cnt, initial=0))
print("s =", s)
for t in range(len(cnt)):
    print(f"loại {t}: viên kẹo số {s[t]} .. {s[t+1]-1}  ({cnt[t]} viên)")

s = [0, 7, 11, 16, 19, 27]
loại 0: viên kẹo số 0 .. 6  (7 viên)
loại 1: viên kẹo số 7 .. 10  (4 viên)
loại 2: viên kẹo số 11 .. 15  (5 viên)
loại 3: viên kẹo số 16 .. 18  (3 viên)
loại 4: viên kẹo số 19 .. 26  (8 viên)


### 3.2 Hai khoảng đóng có giao nhau không?

`[a, b]` và `[c, d]` (với `a ≤ b`, `c ≤ d`) giao nhau ⇔ `a ≤ d and c ≤ b`. Ngược lại là "một khoảng nằm hẳn bên trái khoảng kia".

In [3]:
def overlap(a, b, c, d):
    return a <= d and c <= b

for seg in [((0, 3), (3, 5)), ((0, 2), (3, 5)), ((4, 9), (1, 4)), ((2, 2), (0, 10))]:
    print(seg, "->", overlap(*seg[0], *seg[1]))

((0, 3), (3, 5)) -> True
((0, 2), (3, 5)) -> False
((4, 9), (1, 4)) -> True
((2, 2), (0, 10)) -> True


### 3.3 Ngày đánh số từ 0: "trước ngày `d`" có đúng `d` ngày

Các ngày `0, 1, …, d-1` là **`d` ngày** đã qua trước ngày `d`. Đến hết ngày `d` là **`d + 1` ngày**. Đây là nguồn gốc của rất nhiều lỗi lệch-1 trong bài này.

In [4]:
d = 2
print("trước ngày", d, "đã qua:", list(range(d)), "=", d, "ngày")
print("tính cả ngày", d, ":", list(range(d + 1)), "=", d + 1, "ngày")

trước ngày 2 đã qua: [0, 1] = 2 ngày
tính cả ngày 2 : [0, 1, 2] = 3 ngày


### 3.4 `bisect_right` để tìm "viên kẹo số `k` thuộc loại nào"

Trên mảng tăng `s`, `bisect_right(s, k) - 1` là chỉ số `t` lớn nhất với `s[t] ≤ k` — chính là loại chứa viên kẹo số `k`.

In [5]:
s = list(accumulate([7, 4, 5, 3, 8], initial=0))
for k in [0, 6, 7, 10, 11, 18, 26]:
    print(f"viên số {k:>2} thuộc loại {bisect_right(s, k) - 1}")

viên số  0 thuộc loại 0
viên số  6 thuộc loại 0
viên số  7 thuộc loại 1
viên số 10 thuộc loại 1
viên số 11 thuộc loại 2
viên số 18 thuộc loại 3
viên số 26 thuộc loại 4


### 3.5 Số nguyên lớn trong Python

`(day + 1) * cap` có thể tới `10⁹ · 10⁹ = 10¹⁸`. Python có số nguyên không giới hạn nên không tràn số; nhưng ở C++/Java phải dùng `long long`/`long`.

In [6]:
day, cap = 10**9, 10**9
print((day + 1) * cap, "| vượt int32?", (day + 1) * cap > 2**31 - 1)

1000000001000000000 | vượt int32? True


## 4. Ý tưởng nền tảng / trực giác

**Đánh số viên kẹo.** Vì phải ăn hết loại 0 rồi mới tới loại 1…, thứ tự ăn kẹo là **cố định**: viên số `0, 1, 2, …, total-1`. Chỉ có **tốc độ** là ta được chọn. Loại `t` là đoạn viên số `[s[t], s[t+1] - 1]`.

**Trước ngày `d`, đã ăn bao nhiêu viên?** Có `d` ngày trôi qua, mỗi ngày `1..cap` viên ⇒ đã ăn `e` viên với `d ≤ e ≤ d·cap` (nếu chưa hết kẹo).

**Ngày `d` ăn những viên nào?** Ăn tiếp `k ∈ [1, cap]` viên: viên số `e, …, e + k - 1`.
- Viên **sớm nhất** có thể ăn vào ngày `d`: `e = d`, viên số `d` (ăn chậm nhất: 1 viên/ngày).
- Viên **muộn nhất** có thể ăn vào ngày `d`: `e = d·cap`, `k = cap` ⇒ viên số `(d+1)·cap - 1` (ăn nhanh nhất).

Tính tay Ví dụ 1, truy vấn `[4, 2, 4]`: ngày 2 ăn được viên số `[2, 3·4 - 1] = [2, 11]`. Loại 4 là viên số `[19, 26]`. Không giao ⇒ `false`. ✔

Truy vấn `[2, 13, 10⁹]`: ngày 13 ăn được viên `[13, …rất lớn]`; loại 2 là `[11, 15]` ⇒ giao ⇒ `true`. ✔

In [7]:
cnt = [7, 4, 5, 3, 8]
s = list(accumulate(cnt, initial=0))
for t, day, cap in [[0, 2, 2], [4, 2, 4], [2, 13, 10**9]]:
    lo, hi = day, (day + 1) * cap - 1
    print(f"query {[t, day, cap]}: ngày {day} ăn được viên [{lo}, {hi}], loại {t} = [{s[t]}, {s[t+1]-1}]"
          f" -> giao? {lo <= s[t+1]-1 and s[t] <= hi}")

query [0, 2, 2]: ngày 2 ăn được viên [2, 5], loại 0 = [0, 6] -> giao? True
query [4, 2, 4]: ngày 2 ăn được viên [2, 11], loại 4 = [19, 26] -> giao? False
query [2, 13, 1000000000]: ngày 13 ăn được viên [13, 13999999999], loại 2 = [11, 15] -> giao? True


## 5. Cách 1 — Brute Force / baseline: vét cạn mọi lịch ăn (tập trạng thái)

**Ý tưởng:** trạng thái sau mỗi ngày là "đã ăn tổng cộng `e` viên". Bắt đầu `{0}`. Mỗi ngày, từ `e < total` có thể sang `e + k` với `1 ≤ k ≤ min(cap, total - e)`; nếu `e == total` thì đã hết kẹo (giữ nguyên). Đến ngày `day`, thử mọi `e` và mọi `k`: nếu đoạn viên `[e, e+k-1]` chạm loại `t` thì trả `True`.

Cách này **mô phỏng đúng luật** chứ không dùng công thức, nên là chuẩn đối chiếu đáng tin.

In [8]:
def can_eat_one_brute(cnt, t, day, cap):
    total = sum(cnt)
    s = list(accumulate(cnt, initial=0))
    lo_t, hi_t = s[t], s[t + 1] - 1
    reach = {0}
    for _ in range(day):
        nxt = set()
        for e in reach:
            if e == total:
                nxt.add(e)
            else:
                for k in range(1, min(cap, total - e) + 1):
                    nxt.add(e + k)
        reach = nxt
    for e in reach:
        if e < total:
            for k in range(1, min(cap, total - e) + 1):
                if e <= hi_t and lo_t <= e + k - 1:
                    return True
    return False

def can_eat_brute(cnt, queries):
    return [can_eat_one_brute(cnt, t, d, c) for t, d, c in queries]

for c, q, want in examples:
    q_small = [[t, d, min(cap, 40)] for t, d, cap in q]   # cap lớn hơn tổng kẹo cũng như cap = tổng
    got = can_eat_brute(c, q_small)
    print(got, "khớp:", got == want)

[True, False, True] khớp: True
[False, True, True, False, False] khớp: True


(Ta thay `cap` khổng lồ bằng `min(cap, 40)`: vì tổng kẹo ≤ 27 nên ăn nhiều hơn 27 viên/ngày cũng chẳng khác gì — giảm thời gian vét cạn.)

**Độ phức tạp:** mỗi ngày có tới `total` trạng thái, mỗi trạng thái thử `cap` bước ⇒ `O(day · total · cap)` mỗi truy vấn. Với `day = 10⁹` thì vô vọng.

**Nút thắt:** ta liệt kê từng trạng thái, trong khi tập `reach` thực ra luôn là **một khoảng liên tục** `[d, min(d·cap, total)]`. Kiểm chứng:

In [9]:
def reach_sets(cnt, day, cap):
    total = sum(cnt); reach = {0}; out = []
    for d in range(day + 1):
        out.append(sorted(reach))
        nxt = set()
        for e in reach:
            if e == total: nxt.add(e)
            else:
                for k in range(1, min(cap, total - e) + 1): nxt.add(e + k)
        reach = nxt
    return out

for d, r in enumerate(reach_sets([3, 2, 4], 5, 3)):
    print(f"trước ngày {d}: đã ăn e ∈ {r}  | liên tục: {r == list(range(r[0], r[-1] + 1))}")

trước ngày 0: đã ăn e ∈ [0]  | liên tục: True
trước ngày 1: đã ăn e ∈ [1, 2, 3]  | liên tục: True
trước ngày 2: đã ăn e ∈ [2, 3, 4, 5, 6]  | liên tục: True
trước ngày 3: đã ăn e ∈ [3, 4, 5, 6, 7, 8, 9]  | liên tục: True
trước ngày 4: đã ăn e ∈ [4, 5, 6, 7, 8, 9]  | liên tục: True
trước ngày 5: đã ăn e ∈ [5, 6, 7, 8, 9]  | liên tục: True


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** tập viên kẹo có thể ăn vào ngày `d` (với trần `cap`) là **khoảng** `[d, (d+1)·cap - 1]` (cắt bớt phần vượt `total - 1`).

**Quy tắc "nếu… thì…":**
- *Nếu* `day ≥ s[t+1]` (tức ngay cả ăn chậm nhất 1 viên/ngày thì trước ngày `day` đã ăn ≥ `s[t+1]` viên — hết loại `t`) *thì* `false`.
- *Nếu* `(day+1)·cap ≤ s[t]` (ăn nhanh nhất tới hết ngày `day` vẫn chưa chạm viên số `s[t]`) *thì* `false`.
- Ngược lại `true`.

Gộp lại: `answer = day < s[t+1] and (day + 1) * cap > s[t]`. Đó chính là điều kiện giao của `[day, (day+1)·cap - 1]` và `[s[t], s[t+1] - 1]`.

**Invariant cần nhớ:** "đã ăn trước ngày `d`" ∈ `[d, d·cap]` và mọi số nguyên trong khoảng đó đều đạt được (tăng từng viên một ở một ngày bất kỳ vẫn hợp lệ).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mỗi truy vấn cộng dồn lại từ đầu (`O(n)` mỗi truy vấn)

Đã có công thức, nhưng người mới thường tính `s[t]` và `s[t+1]` bằng vòng lặp cho **mỗi** truy vấn.

In [10]:
def can_eat_rescan(cnt, queries):
    ans = []
    for t, day, cap in queries:
        before = 0
        for i in range(t):          # tổng kẹo các loại trước t: O(t)
            before += cnt[i]
        through = before + cnt[t]   # tổng tới hết loại t
        ans.append(day < through and (day + 1) * cap > before)
    return ans

for c, q, want in examples:
    got = can_eat_rescan(c, q)
    print(got, got == want)

[True, False, True] True
[False, True, True, False, False] True


**Độ phức tạp:** `O(n)` mỗi truy vấn ⇒ `O(n·q)` = `10¹⁰` ở worst case. Nút thắt: tính lại cùng một tổng nhiều lần ⇒ **tính trước một lần** bằng tổng tiền tố.

### 7.2 Cách 3 — Tổng tiền tố + giao khoảng (`O(n + q)`) — lời giải chuẩn

1. `s = [0, c0, c0+c1, …]` (độ dài `n+1`).
2. Với mỗi truy vấn: `ans = day < s[t+1] and (day+1)*cap > s[t]`.

Code demo in ra hai khoảng:

In [11]:
def can_eat_demo(cnt, queries):
    s = list(accumulate(cnt, initial=0))
    ans = []
    for t, day, cap in queries:
        eat_lo, eat_hi = day, (day + 1) * cap - 1
        typ_lo, typ_hi = s[t], s[t + 1] - 1
        ok = eat_lo <= typ_hi and typ_lo <= eat_hi
        print(f"{[t, day, cap]}: ngày ăn được [{eat_lo}, {eat_hi}] vs loại [{typ_lo}, {typ_hi}] -> {ok}")
        ans.append(ok)
    return ans

can_eat_demo(*examples[1][:2])

[3, 1, 2]: ngày ăn được [1, 3] vs loại [13, 16] -> False
[4, 10, 3]: ngày ăn được [10, 32] vs loại [17, 17] -> True
[3, 10, 100]: ngày ăn được [10, 1099] vs loại [13, 16] -> True
[4, 100, 30]: ngày ăn được [100, 3029] vs loại [17, 17] -> False
[1, 3, 1]: ngày ăn được [3, 3] vs loại [5, 6] -> False


[False, True, True, False, False]

In [12]:
class SolutionPrefix:
    def canEat(self, candiesCount: List[int], queries: List[List[int]]) -> List[bool]:
        s = list(accumulate(candiesCount, initial=0))
        return [day < s[t + 1] and (day + 1) * cap > s[t] for t, day, cap in queries]

for c, q, want in examples:
    print(SolutionPrefix().canEat(c, q) == want)

True
True


**Độ phức tạp:** `O(n)` dựng tiền tố + `O(1)` mỗi truy vấn ⇒ `O(n + q)` thời gian, `O(n)` bộ nhớ phụ (cộng `O(q)` cho mảng kết quả).

### 7.3 Cách 4 — Tìm kiếm nhị phân: loại sớm nhất & muộn nhất có thể ăn ngày `day`

Góc nhìn khác: ngày `day` ăn được các viên `[day, (day+1)·cap - 1]`. Viên `day` thuộc loại `lo_type`, viên `min((day+1)·cap - 1, total - 1)` thuộc loại `hi_type`. Truy vấn đúng ⇔ `day < total` và `lo_type ≤ t ≤ hi_type`. Dùng `bisect_right` trên `s` để tìm loại.

In [13]:
class SolutionBisect:
    def canEat(self, candiesCount: List[int], queries: List[List[int]]) -> List[bool]:
        s = list(accumulate(candiesCount, initial=0))
        total = s[-1]
        ans = []
        for t, day, cap in queries:
            if day >= total:                 # đã chắc chắn ăn hết kẹo trước ngày này
                ans.append(False)
                continue
            last = min((day + 1) * cap - 1, total - 1)
            lo_type = bisect_right(s, day) - 1
            hi_type = bisect_right(s, last) - 1
            ans.append(lo_type <= t <= hi_type)
        return ans

for c, q, want in examples:
    print(SolutionBisect().canEat(c, q) == want)

True
True


**Độ phức tạp:** `O(n + q log n)`. Chậm hơn Cách 3 một chút nhưng trả lời được câu hỏi giàu thông tin hơn: "ngày `day` có thể ăn **những loại nào**" — hữu ích cho các biến thể.

## 8. TRACE từng bước

Trace Cách 3 trên Ví dụ 2: `candiesCount = [5,2,6,4,1]` ⇒ `s = [0,5,7,13,17,18]`.

| truy vấn `[t,day,cap]` | `s[t]` | `s[t+1]` | `day < s[t+1]`? | `(day+1)·cap` | `> s[t]`? | kết quả |
|---|---|---|---|---|---|---|
| `[3,1,2]` | 13 | 17 | 1 < 17 ✔ | 4 | 4 > 13 ✘ | false |
| `[4,10,3]` | 17 | 18 | 10 < 18 ✔ | 33 | 33 > 17 ✔ | true |
| `[3,10,100]` | 13 | 17 | 10 < 17 ✔ | 1100 | ✔ | true |
| `[4,100,30]` | 17 | 18 | 100 < 18 ✘ | — | — | false |
| `[1,3,1]` | 5 | 7 | 3 < 7 ✔ | 4 | 4 > 5 ✘ | false |

Truy vấn cuối: ăn 1 viên/ngày thì tới hết ngày 3 mới ăn 4 viên (số 0..3) — chưa tới loại 1 (viên số 5, 6). Với `cap = 1` không thể nhanh hơn ⇒ `false`.

In [14]:
def trace_can_eat(cnt, queries):
    s = list(accumulate(cnt, initial=0))
    print("s =", s)
    res = []
    for t, day, cap in queries:
        c1 = day < s[t + 1]
        most = (day + 1) * cap
        c2 = most > s[t]
        print(f"[{t},{day},{cap}]: s[t]={s[t]:>2}, s[t+1]={s[t+1]:>2} | day<s[t+1]: {c1!s:5} | "
              f"(day+1)*cap={most:>5} > s[t]: {c2!s:5} -> {c1 and c2}")
        res.append(c1 and c2)
    return res

trace_can_eat(*examples[1][:2])

s = [0, 5, 7, 13, 17, 18]
[3,1,2]: s[t]=13, s[t+1]=17 | day<s[t+1]: True  | (day+1)*cap=    4 > s[t]: False -> False
[4,10,3]: s[t]=17, s[t+1]=18 | day<s[t+1]: True  | (day+1)*cap=   33 > s[t]: True  -> True
[3,10,100]: s[t]=13, s[t+1]=17 | day<s[t+1]: True  | (day+1)*cap= 1100 > s[t]: True  -> True
[4,100,30]: s[t]=17, s[t+1]=18 | day<s[t+1]: False | (day+1)*cap= 3030 > s[t]: True  -> False
[1,3,1]: s[t]= 5, s[t+1]= 7 | day<s[t+1]: True  | (day+1)*cap=    4 > s[t]: False -> False


[False, True, True, False, False]

Trace vét cạn cho một truy vấn nhỏ để thấy "lịch ăn" thật sự: `cnt = [3, 2, 4]`, truy vấn `[1, 2, 2]` (loại 1 = viên số 3, 4).

In [15]:
def trace_brute(cnt, t, day, cap):
    s = list(accumulate(cnt, initial=0)); total = s[-1]
    reach = {0}
    for d in range(day):
        nxt = set()
        for e in reach:
            if e == total: nxt.add(e)
            else:
                for k in range(1, min(cap, total - e) + 1): nxt.add(e + k)
        reach = nxt
        print(f"hết ngày {d}: đã ăn e ∈ {sorted(reach)}")
    for e in sorted(reach):
        for k in range(1, min(cap, total - e) + 1):
            hit = e <= s[t + 1] - 1 and s[t] <= e + k - 1
            print(f"ngày {day}: từ e={e} ăn {k} viên -> viên [{e}, {e+k-1}] {'CHẠM loại ' + str(t) if hit else ''}")

trace_brute([3, 2, 4], 1, 2, 2)

hết ngày 0: đã ăn e ∈ [1, 2]
hết ngày 1: đã ăn e ∈ [2, 3, 4]
ngày 2: từ e=2 ăn 1 viên -> viên [2, 2] 
ngày 2: từ e=2 ăn 2 viên -> viên [2, 3] CHẠM loại 1
ngày 2: từ e=3 ăn 1 viên -> viên [3, 3] CHẠM loại 1
ngày 2: từ e=3 ăn 2 viên -> viên [3, 4] CHẠM loại 1
ngày 2: từ e=4 ăn 1 viên -> viên [4, 4] CHẠM loại 1
ngày 2: từ e=4 ăn 2 viên -> viên [4, 5] CHẠM loại 1


## 9. Code nộp LeetCode

In [16]:
from itertools import accumulate
from typing import List

class Solution:
    def canEat(self, candiesCount: List[int], queries: List[List[int]]) -> List[bool]:
        s = list(accumulate(candiesCount, initial=0))   # s[t] = số viên trước loại t
        ans = []
        for t, day, cap in queries:
            # ngày `day` ăn được các viên số [day, (day+1)*cap - 1]
            ans.append(day < s[t + 1] and (day + 1) * cap > s[t])
        return ans

for c, q, want in examples:
    assert Solution().canEat(c, q) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `E_d` = số viên đã ăn trước ngày `d` (sau các ngày `0..d-1`).

**Bổ đề 1:** nếu chưa hết kẹo thì `d ≤ E_d ≤ d·cap`, và mọi số nguyên trong `[d, min(d·cap, total)]` đều đạt được. *Chứng minh:* mỗi ngày ăn từ 1 đến `cap` viên nên chặn dưới/trên hiển nhiên. Để đạt `E_d = m` với `d ≤ m ≤ d·cap`: cho mỗi ngày ăn `⌊m/d⌋` hoặc `⌈m/d⌉` viên — mỗi giá trị nằm trong `[1, cap]`.

**Bổ đề 2:** tập viên có thể ăn vào ngày `d` là `[d, min((d+1)·cap, total) - 1]`. *Chứng minh:* ngày `d` ăn viên `E_d, …, E_d + k - 1` với `1 ≤ k ≤ cap`. Hợp các đoạn này khi `E_d` chạy trên `[d, d·cap]` và `k` trên `[1, cap]` là đoạn liên tục từ `d` tới `d·cap + cap - 1` (các đoạn chồng lên nhau vì bước `E_d` là 1).

**Kết luận:** ăn được loại `t` ngày `d` ⇔ đoạn ở Bổ đề 2 giao `[s[t], s[t+1]-1]` ⇔ `d ≤ s[t+1] - 1` và `s[t] ≤ (d+1)·cap - 1` ⇔ `d < s[t+1]` và `(d+1)·cap > s[t]`. (Điều kiện `d < s[t+1] ≤ total` cũng bảo đảm ngày `d` còn kẹo.)

## 11. Phân tích Time / Space

Gọi `n = len(candiesCount)`, `q = len(queries)`, `D = favoriteDay`, `C = dailyCap`, `T = tổng kẹo`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Vét cạn trạng thái | `O(q · D · T · C)` | `O(T)` |
| 2. Cộng dồn mỗi truy vấn | `O(n · q)` | `O(1)` |
| 3. Tổng tiền tố | `O(n + q)` | `O(n)` |
| 4. Tiền tố + bisect | `O(n + q log n)` | `O(n)` |

Không có khác biệt average/worst: mọi cách đều tất định. Đo thực nghiệm ở kích thước tối đa:

In [17]:
rng = random.Random(1744)
n = q = 10**5
cnt_big = [rng.randint(1, 10**5) for _ in range(n)]
qs_big = [[rng.randrange(n), rng.randint(0, 10**9), rng.randint(1, 10**9)] for _ in range(q)]
for name, sol in [("Cách 3", SolutionPrefix()), ("Cách 4", SolutionBisect()), ("Nộp", Solution())]:
    t0 = time.perf_counter()
    r = sol.canEat(cnt_big, qs_big)
    print(f"{name}: {1000*(time.perf_counter()-t0):.1f} ms, số True = {sum(r)}")
assert SolutionPrefix().canEat(cnt_big, qs_big) == SolutionBisect().canEat(cnt_big, qs_big)
t0 = time.perf_counter()
can_eat_rescan(cnt_big, qs_big[:200])
dt = time.perf_counter() - t0
print(f"Cách 2 với 200 truy vấn: {dt*1000:.1f} ms -> ước tính 10^5 truy vấn ≈ {dt*500:.0f} s")

Cách 3: 23.7 ms, số True = 89845


Cách 4: 49.0 ms, số True = 89845


Nộp: 20.1 ms, số True = 89845


Cách 2 với 200 truy vấn: 246.0 ms -> ước tính 10^5 truy vấn ≈ 123 s


## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đã đối chiếu vét cạn)

- Ngày 0 (chưa ăn gì): loại 0 luôn `true`; loại 1 `true` chỉ khi `cap > s[1]`.
- `day` đúng bằng `s[t+1]`: đã hết loại `t` ⇒ `false`.
- `day ≥ total`: hết kẹo ⇒ `false`.
- `cap = 1`: chỉ có một lịch ăn duy nhất.

In [18]:
sols = [Solution(), SolutionPrefix(), SolutionBisect()]
edge = [
    ([1, 1], [[1, 0, 2]], [True]),          # ngày 0 ăn 2 viên -> chạm loại 1
    ([1, 1], [[1, 0, 1]], [False]),         # cap=1: ngày 0 chỉ ăn viên số 0
    ([2], [[0, 2, 1]], [False]),            # day == s[1] = 2: đã hết
    ([2], [[0, 1, 1]], [True]),
    ([3], [[0, 5, 1]], [False]),            # day >= total
    ([5, 5], [[0, 0, 1], [1, 4, 1], [1, 5, 1], [1, 9, 1], [1, 10, 1]], [True, False, True, True, False]),
    ([10**5], [[0, 10**9, 10**9]], [False]),
    ([10**5] * 3, [[2, 0, 10**9]], [True]),
]
for c, q, want in examples + edge:
    for sol in sols:
        assert sol.canEat(c, q) == want, (c, q, sol)
    small_q = [[t, d, min(cap, sum(c) + 1)] for t, d, cap in q]
    if sum(c) <= 30 and max(d for _, d, _ in q) <= 30:
        assert can_eat_brute(c, small_q) == want
    assert can_eat_rescan(c, q) == want
print("Tất cả edge case OK")

Tất cả edge case OK


### 12.2 Ngẫu nhiên có seed: công thức vs vét cạn mô phỏng luật

Ràng buộc thu nhỏ để vét cạn chạy nhanh: `n ≤ 4`, mỗi loại ≤ 4 viên, `day ≤ 12`, `cap ≤ 5` (vẫn nằm trong ràng buộc đề).

In [19]:
rng = random.Random(42)
checked = 0
for _ in range(400):
    c = [rng.randint(1, 4) for _ in range(rng.randint(1, 4))]
    q = [[rng.randrange(len(c)), rng.randint(0, 12), rng.randint(1, 5)] for _ in range(4)]
    want = can_eat_brute(c, q)
    for sol in sols:
        assert sol.canEat(c, q) == want, (c, q)
    assert can_eat_rescan(c, q) == want
    checked += len(q)
print("đã đối chiếu", checked, "truy vấn với vét cạn")

đã đối chiếu 1600 truy vấn với vét cạn


### 12.3 Kiểm chứng Bổ đề 2 trực tiếp: tập viên ăn được ngày `d` là một khoảng

In [20]:
def eatable_set(cnt, day, cap):
    total = sum(cnt); reach = {0}
    for _ in range(day):
        nxt = set()
        for e in reach:
            if e == total: nxt.add(e)
            else:
                for k in range(1, min(cap, total - e) + 1): nxt.add(e + k)
        reach = nxt
    out = set()
    for e in reach:
        for k in range(1, min(cap, total - e) + 1):
            out.update(range(e, e + k))
    return out

rng = random.Random(3)
for _ in range(200):
    c = [rng.randint(1, 4) for _ in range(rng.randint(1, 4))]
    total = sum(c); d = rng.randint(0, 10); cap = rng.randint(1, 4)
    got = eatable_set(c, d, cap)
    want = set(range(d, min((d + 1) * cap, total))) if d < total else set()
    assert got == want
print("Bổ đề 2 đúng trên 200 case ngẫu nhiên")

Bổ đề 2 đúng trên 200 case ngẫu nhiên


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Quên ngày 0: dùng `day * cap` thay vì `(day + 1) * cap`

In [21]:
def wrong_off_by_one(cnt, queries):
    s = list(accumulate(cnt, initial=0))
    return [day < s[t + 1] and day * cap > s[t] for t, day, cap in queries]   # SAI

c, q = [1, 1], [[1, 0, 2]]
print("sai:", wrong_off_by_one(c, q), "| vét cạn:", can_eat_brute(c, q))

sai: [False] | vét cạn: [True]


**Vì sao sai:** đến **hết** ngày `day` đã có `day + 1` ngày (0..day). Ngày 0 ăn 2 viên là chạm loại 1 rồi, nhưng `0 * 2 = 0` khiến công thức nói không. **Sửa:**

In [22]:
print("đúng:", Solution().canEat(c, q))

đúng: [True]


### Lỗi 2 — Dùng `<=` thay vì `<` ở điều kiện chậm nhất

In [23]:
def wrong_le(cnt, queries):
    s = list(accumulate(cnt, initial=0))
    return [day <= s[t + 1] and (day + 1) * cap > s[t] for t, day, cap in queries]   # SAI

c, q = [2], [[0, 2, 1]]
print("sai:", wrong_le(c, q), "| vét cạn:", can_eat_brute(c, q))
print("đúng:", Solution().canEat(c, q))

sai: [True] | vét cạn: [False]
đúng: [False]


**Vì sao sai:** trước ngày 2 đã qua 2 ngày, mỗi ngày ≥ 1 viên ⇒ đã ăn ≥ 2 = `s[1]` viên, tức đã hết loại 0. Viên sớm nhất ngày `day` là viên số `day`, phải `≤ s[t+1] - 1` ⇔ `day < s[t+1]`.

### Lỗi 3 — Quên luật "mỗi ngày ít nhất 1 viên" (coi như được nhịn ăn)

In [24]:
def wrong_can_skip(cnt, queries):
    s = list(accumulate(cnt, initial=0))
    # SAI: nghĩ rằng có thể ăn 0 viên/ngày nên "chậm nhất" là chưa ăn gì
    return [(day + 1) * cap > s[t] for t, day, cap in queries]

c, q = [3], [[0, 5, 1]]
print("sai:", wrong_can_skip(c, q), "| vét cạn:", can_eat_brute(c, q))
print("đúng:", Solution().canEat(c, q))

sai: [True] | vét cạn: [False]
đúng: [False]


**Vì sao sai:** bắt buộc ăn ≥ 1 viên mỗi ngày nên sau 3 ngày đã hết 3 viên; ngày 5 không còn kẹo. Luật này chính là nguồn gốc vế `day < s[t+1]`.

### Lỗi 4 — Tổng tiền tố thiếu `initial=0` rồi dùng chỉ số như có

In [25]:
def wrong_prefix_index(cnt, queries):
    s = list(accumulate(cnt))          # s[i] = tổng tới HẾT loại i (không có số 0 đầu)
    return [day < s[t + 1] and (day + 1) * cap > s[t] if t + 1 < len(s) else False
            for t, day, cap in queries]   # SAI: lệch một loại

c, q = [7, 4, 5, 3, 8], [[0, 2, 2], [4, 2, 4], [2, 13, 10**9]]
print("sai:", wrong_prefix_index(c, q), "| đúng:", Solution().canEat(c, q))

sai: [False, False, True] | đúng: [True, False, True]


**Vì sao sai:** không có `initial=0` thì `s[t]` là tổng **tới hết** loại `t` (chứ không phải trước loại `t`); mọi so sánh bị lệch sang loại kế tiếp, và loại cuối còn vượt chỉ số. **Sửa:** dùng `accumulate(cnt, initial=0)` hoặc tự thêm `[0] +`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Ăn/xử lý theo thứ tự cố định, chỉ chọn tốc độ"** ⇒ đánh số phần tử toàn cục, biến bài thành **khoảng số**.
- **Nhiều truy vấn trên đoạn liên tiếp** ⇒ tổng tiền tố, `O(1)` mỗi truy vấn.
- **Có chặn dưới/chặn trên mỗi bước** ⇒ xét hai cực trị (chậm nhất, nhanh nhất); nếu bước nhảy là 1 thì mọi giá trị giữa đều đạt được.
- **Giao hai khoảng đóng** `[a,b]∩[c,d]≠∅ ⇔ a ≤ d and c ≤ b` — viết thẳng ra rồi biến đổi để tránh nhầm `<`/`≤`.
- Ngày đánh số từ 0: "trước ngày `d`" = `d` ngày; "tới hết ngày `d`" = `d+1` ngày.

## 15. Biến thể của bài

1. **Có chặn dưới `minCap` mỗi ngày** (thay cho 1): khoảng ăn được ngày `d` thành `[d·minCap, (d+1)·cap - 1]` (xem Bài tập 5).
2. **Hỏi "những loại nào ăn được ngày `d`"**: dùng Cách 4 (bisect) trả về đoạn loại `[lo_type, hi_type]`.
3. **Hỏi ngày sớm nhất/muộn nhất ăn được loại `t`**: sớm nhất `⌊s[t] / cap⌋`, muộn nhất `s[t+1] - 1`.
4. **Truy vấn có cập nhật số kẹo** (`candiesCount[i]` thay đổi): thay tổng tiền tố bằng **Fenwick tree / Segment tree**.
5. Bài cùng tư duy "mọi giá trị giữa min và max đều đạt được": LeetCode 1798 *Maximum Number of Consecutive Values You Can Make*, 2241 *Design an ATM Machine* (một phần).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn trạng thái | mô phỏng mọi lịch ăn | `O(q·D·T·C)` | `O(T)` | kiểm chứng công thức, input tí hon |
| 2. Cộng dồn mỗi truy vấn | công thức đúng nhưng tính tổng lại | `O(n·q)` | `O(1)` | `q` rất nhỏ |
| 3. Tiền tố + giao khoảng | `day < s[t+1] and (day+1)·cap > s[t]` | `O(n+q)` | `O(n)` | **nộp bài** |
| 4. Tiền tố + bisect | tìm loại sớm/muộn nhất của ngày | `O(n+q log n)` | `O(n)` | cần biết cả dải loại |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `candiesCount = [3, 2]`, truy vấn `[1, 1, 2]`. Kết quả là gì?

**Bài 2 (tính tay):** Với Ví dụ 2, truy vấn `[4, 10, 3]`: tính khoảng viên ăn được ngày 10 và khoảng viên của loại 4, rồi kết luận.

**Bài 3 (điền TODO):** Viết `eat_range(day, cap, total)` trả về `(lo, hi)` — khoảng số thứ tự viên có thể ăn vào ngày `day` (đã cắt theo `total`), hoặc `None` nếu đã hết kẹo.

**Bài 4 (tự viết lại):** Viết `earliest_day(cnt, t, cap)` — ngày **sớm nhất** có thể ăn loại `t` với trần `cap`, và `latest_day(cnt, t)` — ngày **muộn nhất** (không phụ thuộc `cap`).

**Bài 5 (mini challenge):** Biến thể: mỗi ngày phải ăn **ít nhất `a`** viên và **nhiều nhất `b`** viên (cho tới khi số kẹo còn lại < `a` thì ăn hết phần còn lại trong ngày đó). Viết `can_eat_ab(cnt, t, day, a, b)` và đối chiếu với vét cạn.

In [26]:
def eat_range(day, cap, total):
    # TODO: trả về (lo, hi) hoặc None
    pass

def earliest_day(cnt, t, cap):
    # TODO
    pass

def latest_day(cnt, t):
    # TODO
    pass

def can_eat_ab(cnt, t, day, a, b):
    # TODO: biến thể chặn dưới a, chặn trên b
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Loại 1 là viên số `[3, 4]` | Ngày 1 ăn được viên `[1, 2·2-1]` | `[1,3]` giao `[3,4]` |
| 2 | `s = [0,5,7,13,17,18]` | Ngày 10: `[10, 32]` | Loại 4: `[17, 17]` |
| 3 | Nếu `day ≥ total` thì hết | `hi = min((day+1)*cap, total) - 1` | `lo = day` |
| 4 | Sớm nhất: ăn nhanh nhất để tới viên `s[t]` | Viên `s[t]` được ăn ngày `s[t] // cap` | Muộn nhất: 1 viên/ngày tới viên `s[t+1]-1` |
| 5 | Trước ngày `d` đã ăn trong `[d·a, d·b]` (nếu chưa hết) | Viên sớm nhất ngày `d` là `d·a` | Ăn được ⇔ `d·a < s[t+1]` và `(d+1)·b > s[t]` |

## 19. Đáp án tham khảo

In [27]:
# Bài 1 & 2
print("Bài 1:", Solution().canEat([3, 2], [[1, 1, 2]]), "| vét cạn:", can_eat_brute([3, 2], [[1, 1, 2]]))
assert Solution().canEat([3, 2], [[1, 1, 2]]) == [True]
s2 = list(accumulate([5, 2, 6, 4, 1], initial=0))
print("Bài 2: ngày 10 ăn được", (10, 11 * 3 - 1), "| loại 4 =", (s2[4], s2[5] - 1))
assert 10 <= s2[5] - 1 and s2[4] <= 32

Bài 1: [True] | vét cạn: [True]
Bài 2: ngày 10 ăn được (10, 32) | loại 4 = (17, 17)


In [28]:
def eat_range(day, cap, total):
    if day >= total:
        return None
    return day, min((day + 1) * cap, total) - 1

def earliest_day(cnt, t, cap):
    s = list(accumulate(cnt, initial=0))
    return s[t] // cap

def latest_day(cnt, t):
    s = list(accumulate(cnt, initial=0))
    return s[t + 1] - 1

rng = random.Random(8)
for _ in range(150):
    c = [rng.randint(1, 4) for _ in range(rng.randint(1, 4))]
    total = sum(c); cap = rng.randint(1, 4); d = rng.randint(0, 12)
    es = eatable_set(c, d, cap)
    er = eat_range(d, cap, total)
    assert (es == set()) if er is None else (es == set(range(er[0], er[1] + 1)))
    t = rng.randrange(len(c))
    days_ok = [dd for dd in range(total + 2) if can_eat_one_brute(c, t, dd, cap)]
    assert days_ok[0] == earliest_day(c, t, cap) and days_ok[-1] == latest_day(c, t)
    assert days_ok == list(range(days_ok[0], days_ok[-1] + 1))
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [29]:
def can_eat_ab(cnt, t, day, a, b):
    s = list(accumulate(cnt, initial=0))
    return day * a < s[t + 1] and (day + 1) * b > s[t]

def can_eat_ab_brute(cnt, t, day, a, b):
    s = list(accumulate(cnt, initial=0)); total = s[-1]
    def steps(e):
        rem = total - e
        if rem == 0: return []
        if rem < a: return [rem]                 # còn ít hơn a: ăn hết trong ngày
        return list(range(a, min(b, rem) + 1))
    reach = {0}
    for _ in range(day):
        reach = {e + k for e in reach for k in (steps(e) or [0])}
    return any(e <= s[t + 1] - 1 and s[t] <= e + k - 1 for e in reach for k in steps(e))

rng = random.Random(15)
for _ in range(400):
    c = [rng.randint(1, 5) for _ in range(rng.randint(1, 4))]
    a = rng.randint(1, 3); b = rng.randint(a, 5)
    t = rng.randrange(len(c)); d = rng.randint(0, 10)
    assert can_eat_ab(c, t, d, a, b) == can_eat_ab_brute(c, t, d, a, b), (c, t, d, a, b)
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu dùng đề khác): OK so với vét cạn")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu dùng đề khác): OK so với vét cạn


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
s = list(accumulate(candiesCount, initial=0))      # loại t = viên [s[t], s[t+1]-1]
for t, day, cap in queries:
    # ngày day ăn được viên [day, (day+1)*cap - 1]
    ans.append(day < s[t + 1] and (day + 1) * cap > s[t])
```

**Checklist:**
- [ ] Tổng tiền tố có `initial=0`, độ dài `n + 1`?
- [ ] Chậm nhất: `day < s[t+1]` (dấu `<`, không phải `≤`)?
- [ ] Nhanh nhất: `(day + 1) * cap` (nhớ cộng 1 cho ngày 0) và so sánh `> s[t]`?
- [ ] Đọc đúng thứ tự trường truy vấn `[type, day, cap]`?
- [ ] Ở ngôn ngữ khác: dùng số 64-bit cho `(day+1)*cap` (tới `10¹⁸`).
- [ ] Độ phức tạp `O(n + q)`, không vòng lặp theo ngày.